In [1]:
!pip install -q -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.6 MB/s eta 0:00:00


In [3]:
from google.colab import drive, userdata
from huggingface_hub import login
drive.mount("/content/drive")
login(userdata.get("HF_TOKEN"))

Mounted at /content/drive


In [4]:
PROMPT_ID = "caption_v1"
PROMPT = (
    "Write a figure caption for this medical image as it would appear in a radiology "
    "case report. State the imaging modality and anatomical region, and describe visible "
    "findings. Do not speculate about the diagnosis. Respond in one or two sentences."
)
MAX_NEW_TOKENS = 100
N_IMAGES = None

MODEL_NAME = "llava-med-v1.5-mistral-7b-hf"

DRIVE = "/content/drive/MyDrive/vlm-caption-bench"
SUBSET_FILE = f"{DRIVE}/subset_2000.csv"
OUT_DIR = f"{DRIVE}/predictions"

In [5]:
import os, zipfile
import pandas as pd
from pathlib import Path
from PIL import Image

os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.isdir("test"):
    with zipfile.ZipFile(f"{DRIVE}/test.zip") as z:
        z.extractall("test")

ids = pd.read_csv(SUBSET_FILE)["ID"].astype(str).tolist()
if N_IMAGES:
    ids = ids[:N_IMAGES]
by_stem = {p.stem: p for p in Path("test").rglob("*.jpg")}
missing = [i for i in ids if i not in by_stem]
assert not missing, f"немає файлів для {len(missing)} ID, напр. {missing[:3]}"
paths = [by_stem[i] for i in ids]
print(len(paths), "images")

2000 images


In [6]:
import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_path = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
model = LlavaForConditionalGeneration.from_pretrained(
    model_path, quantization_config=bnb, device_map="auto")
processor = AutoProcessor.from_pretrained(model_path)

config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/70.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/3.51M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

In [7]:
def caption(image, prompt=PROMPT):
    text = f"[INST] <image>\n{prompt} [/INST]"
    inputs = processor(images=image, text=text, return_tensors="pt").to(model.device, torch.float16)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [8]:
for p in paths[:3]:
    print(p.name, "->", caption(Image.open(p).convert("RGB")), "\n")

ROCOv2_2023_test_005382.jpg -> The image is a T2-weighted MRI of the cervical spine. The visible findings in the image include a hyperintense lesion in the right posterior cervical spine, which is causing compression of the spinal cord. 

ROCOv2_2023_test_004808.jpg -> The image is a chest X-ray, which is a common imaging modality used to visualize the structures within the chest, including the lungs, heart, and bones of the chest and spine. In this particular image, there are visible findings, but without more information or the ability to view the image, it is not possible to provide specific details about the findings. It is important to consult a healthcare professional for a thorough evaluation and proper diagnosis of the underlying cause of these findings. 

ROCOv2_2023_test_002741.jpg -> The image is a transverse ultrasound of the abdomen. It shows a large, well-defined, heterogeneous mass with internal septations and calcifications. The mass is located in the right upper quadra

In [11]:
import json, os, time, torch

OUT = Path(OUT_DIR) / f"{MODEL_NAME}__{PROMPT_ID}.jsonl"

def read_done(path):
    ids, good = set(), []
    if path.exists():
        for l in path.open(encoding="utf-8"):
            try:
                r = json.loads(l)
            except json.JSONDecodeError:
                continue
            ids.add(r["image_id"])
            good.append(l if l.endswith("\n") else l + "\n")
        path.write_text("".join(good), encoding="utf-8")
    return ids

done = read_done(OUT)
print(len(paths), "images,", len(done), "already done")

failed = []
with OUT.open("a", encoding="utf-8") as f:
    for i, p in enumerate(paths, 1):
        image_id = p.stem
        if image_id in done:
            continue
        try:
            cap = caption(Image.open(p).convert("RGB"))
        except torch.cuda.OutOfMemoryError:
            print(f"OOM on {p.name} - пропущено, буде повторено при наступному запуску")
            failed.append(p.name)
            torch.cuda.empty_cache()
            continue
        rec = {
            "image_id": image_id,
            "caption": cap,
            "model": MODEL_NAME,
            "prompt_id": PROMPT_ID,
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": False
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
        torch.cuda.empty_cache()
        if i % 50 == 0:
            print(f"{i}/{len(paths)} done")
print("failed:", failed)

2000 images, 17 already done
50/2000 done
100/2000 done
150/2000 done
200/2000 done
250/2000 done
300/2000 done
350/2000 done
400/2000 done
450/2000 done
500/2000 done
550/2000 done
600/2000 done
650/2000 done
700/2000 done
750/2000 done
800/2000 done
850/2000 done
900/2000 done
950/2000 done
1000/2000 done
1050/2000 done
1100/2000 done
1150/2000 done
1200/2000 done
1250/2000 done
1300/2000 done
1350/2000 done
1400/2000 done
1450/2000 done
1500/2000 done
1550/2000 done
1600/2000 done
1650/2000 done
1700/2000 done
1750/2000 done
1800/2000 done
1850/2000 done
1900/2000 done
1950/2000 done
2000/2000 done
failed: []
